In [ ]:
PROJECT_REF = "main"


def _install_project(project_ref: str):
    import urllib.parse
    import urllib.request

    from google.colab import userdata

    github_token = userdata.get("GITHUB_TOKEN_JLENS_REAS")
    if not github_token:
        raise RuntimeError(
            "Required Colab secret GITHUB_TOKEN_JLENS_REAS is unavailable"
        )

    query = urllib.parse.urlencode({"ref": project_ref})
    bootstrap_url = (
        "https://api.github.com/repos/noamdwc/jlens-reasoning/"
        "contents/scripts/colab_bootstrap.py?" + query
    )
    request = urllib.request.Request(
        bootstrap_url,
        headers={
            "Authorization": f"Bearer {github_token}",
            "Accept": "application/vnd.github.raw+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    try:
        with urllib.request.urlopen(request) as response:
            bootstrap_source = response.read().decode("utf-8")
    except Exception:
        raise RuntimeError("Unable to load the Colab bootstrap") from None

    namespace = {}
    exec(
        compile(
            bootstrap_source,
            "scripts/colab_bootstrap.py",
            "exec",
        ),
        namespace,
    )
    return namespace["bootstrap"](
        project_ref=project_ref,
        github_token=github_token,
    )


PROJECT_DIR = _install_project(PROJECT_REF)
del _install_project

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
import importlib.metadata
import subprocess

import jlens
import torch
import transformers

from jlens_reasoning.experiments.readout_sanity import (
    LENS_FILE,
    LENS_REPO,
    LENS_REVISION,
    MODEL_NAME,
    READOUT_CASES,
    concept_token_variants,
    run_readout_sanity,
    validate_model_lens,
    write_results,
)

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO,
    filename=LENS_FILE,
    revision=LENS_REVISION,
)
validate_model_lens(model, lens)
model, lens

In [ ]:
result = run_readout_sanity(model=model, lens=lens, tokenizer=tokenizer)
result["provenance"] = {
    "project_commit": subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "jlens": importlib.metadata.version("jlens"),
}

run_dir = context.runs_dir / "jlens-readout-sanity"
result_path = run_dir / "result.json"
write_results(result_path, result)

for case in result["cases"]:
    summary = case["summary"]["jacobian_lens"]
    print(
        case["key"],
        f"baseline={case['baseline']['top1_token']!r}",
        f"best_rank={summary['best_rank']}",
        f"layer={summary['layer']}",
        f"position={summary['position']}",
        f"passed={case['passed']}",
    )
print(f"Saved: {result_path}")

In [ ]:
from IPython.display import display
from jlens.vis import build_page, compute_slice, notebook_iframe

for case in (READOUT_CASES[0], READOUT_CASES[1]):
    pinned = {
        variant.token_id
        for variant in concept_token_variants(tokenizer, case.target_concepts)
    }
    slice_data = compute_slice(
        model,
        lens,
        case.prompt,
        top_n=25,
        pinned_token_ids=pinned,
        mask_display=True,
    )
    page, _, _ = build_page(
        slice_data,
        case.prompt,
        title=f"J-Lens readout sanity: {case.key}",
        description="Readout-only open-model sanity check.",
        mode="embed",
    )
    html_path = run_dir / f"{case.key}.html"
    html_path.write_text(page, encoding="utf-8")
    print(f"Saved: {html_path}")
    display(notebook_iframe(page))

In [ ]:
if not result["passed"]:
    raise RuntimeError(
        "Readout sanity checks failed: " + "; ".join(result["failures"])
    )

print("All J-Lens readout sanity checks passed.")